# NB06: Evidence Integration & Scoring

**Purpose**: Combine all evidence channels from NB02–NB05, assign confidence tiers,
and compute integrated coverage statistics.

**Evidence sources** (by tier):
- **Tier 1** (UniProt-native): EC, BRENDA, Rhea → `uniprot_native_protein_ec.parquet`
- **Tier 2** (Pangenome): eggNOG EC, bakta EC → `pangenome_gc_ec.parquet`
- **Tier 3** (Curated): PaperBLAST, seedclass, besthitmetacyc → `curated_evidence_ec.parquet` + `besthitmetacyc_rxnid.parquet`
- **RAST**: Already processed → `rast_protein_ec.parquet` (used for validation in NB07)

**Strategy**: Each source maps entities (proteins, gene clusters, loci) to ECs.
For reaction coverage, we work at the EC level — the set of unique ECs reached by
each channel, then expand via the EC→reaction bridge. Entity-level integration
(which protein maps to which reaction) requires gene_cluster→protein expansion
and is deferred.

**Output**: `evidence_integration_summary.parquet`, coverage statistics

In [1]:
import os
import pandas as pd
import numpy as np

DATA_DIR = '../data'

ec_bridge = pd.read_parquet(f'{DATA_DIR}/ec_to_reaction.parquet')
kegg_bridge = pd.read_parquet(f'{DATA_DIR}/kegg_to_reaction.parquet')
metacyc_bridge = pd.read_parquet(f'{DATA_DIR}/metacyc_to_reaction.parquet')

balanced_ids = set(
    pd.read_csv(f'{DATA_DIR}/reactions_all.tsv', sep='\t', usecols=['id', 'status'])
    .query("status == 'OK'")['id']
    .str.replace('seed.reaction:', '', regex=False)
)

bridge_ecs = set(ec_bridge['ec'])
bridge_keggs = set(kegg_bridge['kegg_reaction'])
bridge_metacyc = set(metacyc_bridge['metacyc_reaction'])

print(f'Balanced reactions: {len(balanced_ids):,}')
print(f'EC bridge:      {len(ec_bridge):,} mappings, {len(bridge_ecs):,} ECs')
print(f'KEGG bridge:    {len(kegg_bridge):,} mappings, {len(bridge_keggs):,} R-numbers')
print(f'MetaCyc bridge: {len(metacyc_bridge):,} mappings, {len(bridge_metacyc):,} MetaCyc rxns')

Balanced reactions: 34,343
EC bridge:      22,823 mappings, 6,100 ECs
KEGG bridge:    6,851 mappings, 6,354 R-numbers
MetaCyc bridge: 5,039 mappings, 4,480 MetaCyc rxns


## 1. Load All Evidence Sources

In [2]:
tier1 = pd.read_parquet(f'{DATA_DIR}/uniprot_native_protein_ec.parquet')
tier1_ecs = set(tier1['ec'])
tier1_rxns = set(ec_bridge[ec_bridge['ec'].isin(tier1_ecs)]['rxn_bare'])

print(f'Tier 1 (UniProt-native):')
print(f'  Proteins: {tier1.protein.nunique():,}')
print(f'  ECs:      {len(tier1_ecs):,}')
print(f'  Reactions: {len(tier1_rxns):,} / {len(balanced_ids):,} ({100*len(tier1_rxns)/len(balanced_ids):.1f}%)')
print(f'  Channels: {tier1.channels.value_counts().head(5).to_dict()}')

Tier 1 (UniProt-native):


  Proteins: 26,549,024
  ECs:      5,032
  Reactions: 17,215 / 34,343 (50.1%)
  Channels: {'uniprot_ec': 27429790, 'rhea,uniprot_ec': 180501, 'brenda,rhea,uniprot_ec': 11844, 'brenda': 9073, 'brenda,uniprot_ec': 7977}


In [3]:
tier2_ec = pd.read_parquet(f'{DATA_DIR}/pangenome_gc_ec.parquet')
tier2_ecs = set(tier2_ec['ec'])
tier2_ec_rxns = set(ec_bridge[ec_bridge['ec'].isin(tier2_ecs)]['rxn_bare'])

print(f'Tier 2 (Pangenome) — EC channel:')
print(f'  Gene clusters: {tier2_ec.gene_cluster_id.nunique():,}')
print(f'  ECs:           {len(tier2_ecs):,}')
print(f'  Reactions via EC: {len(tier2_ec_rxns):,} / {len(balanced_ids):,} ({100*len(tier2_ec_rxns)/len(balanced_ids):.1f}%)')
print(f'  Channels: {tier2_ec.channels.value_counts().head(5).to_dict()}')

Tier 2 (Pangenome) — EC channel:


  Gene clusters: 25,367,594
  ECs:           3,783
  Reactions via EC: 14,557 / 34,343 (42.4%)
  Channels: {'eggnog_ec': 14747672, 'bakta_ec,eggnog_ec': 11671413, 'bakta_ec': 3761363}


In [4]:
tier3 = pd.read_parquet(f'{DATA_DIR}/curated_evidence_ec.parquet')
tier3_ecs = set(tier3['ec'])
tier3_ec_rxns = set(ec_bridge[ec_bridge['ec'].isin(tier3_ecs)]['rxn_bare'])

bhmc_rxnid = pd.read_parquet(f'{DATA_DIR}/besthitmetacyc_rxnid.parquet')
tier3_metacyc_rxns = set(bhmc_rxnid['rxn_bare'])

tier3_rxns = tier3_ec_rxns | tier3_metacyc_rxns

print(f'Tier 3 (Curated):')
print(f'  Entities: {tier3.protein.nunique():,}')
print(f'  ECs:      {len(tier3_ecs):,}')
print(f'  Reactions (EC):     {len(tier3_ec_rxns):,}')
print(f'  Reactions (rxnId):  {len(tier3_metacyc_rxns):,}')
print(f'  Reactions (total):  {len(tier3_rxns):,} / {len(balanced_ids):,} ({100*len(tier3_rxns)/len(balanced_ids):.1f}%)')
print(f'  Channels: {tier3.channel.value_counts().to_dict()}')

Tier 3 (Curated):
  Entities: 117,294
  ECs:      4,780
  Reactions (EC):     16,254
  Reactions (rxnId):  950
  Reactions (total):  16,290 / 34,343 (47.4%)
  Channels: {'paperblast': 74911, 'seedclass': 49930, 'besthitmetacyc_ec': 25817}


In [5]:
rast = pd.read_parquet(f'{DATA_DIR}/rast_protein_ec.parquet')
rast_ecs = set(rast['ec'])
rast_rxns = set(ec_bridge[ec_bridge['ec'].isin(rast_ecs)]['rxn_bare'])

print(f'RAST (validation set, also an evidence channel):')
print(f'  Proteins: {rast.protein_id.nunique():,}')
print(f'  ECs:      {len(rast_ecs):,}')
print(f'  Reactions: {len(rast_rxns):,} / {len(balanced_ids):,} ({100*len(rast_rxns)/len(balanced_ids):.1f}%)')

RAST (validation set, also an evidence channel):


  Proteins: 30,363,667
  ECs:      2,439
  Reactions: 10,730 / 34,343 (31.2%)


## 2. EC-Level Coverage Analysis

Track which ECs and reactions each tier contributes, including additive value.

In [6]:
all_ecs = tier1_ecs | tier2_ecs | tier3_ecs | rast_ecs

print(f'EC coverage by tier:')
print(f'  Tier 1 (UniProt-native): {len(tier1_ecs):,} unique ECs')
print(f'  Tier 2 (Pangenome):      {len(tier2_ecs):,} unique ECs')
print(f'  Tier 3 (Curated):        {len(tier3_ecs):,} unique ECs')
print(f'  RAST:                    {len(rast_ecs):,} unique ECs')
print(f'  Combined:                {len(all_ecs):,} unique ECs')
print(f'  In bridge:               {len(all_ecs & bridge_ecs):,} / {len(bridge_ecs):,} ({100*len(all_ecs & bridge_ecs)/len(bridge_ecs):.1f}%)')

print(f'\nEC overlap between tiers (intersection sizes):')
print(f'  T1 ∩ T2: {len(tier1_ecs & tier2_ecs):,}')
print(f'  T1 ∩ T3: {len(tier1_ecs & tier3_ecs):,}')
print(f'  T2 ∩ T3: {len(tier2_ecs & tier3_ecs):,}')
print(f'  T1 ∩ T2 ∩ T3: {len(tier1_ecs & tier2_ecs & tier3_ecs):,}')

print(f'\nTier-exclusive ECs:')
print(f'  T1 only: {len(tier1_ecs - tier2_ecs - tier3_ecs):,}')
print(f'  T2 only: {len(tier2_ecs - tier1_ecs - tier3_ecs):,}')
print(f'  T3 only: {len(tier3_ecs - tier1_ecs - tier2_ecs):,}')

EC coverage by tier:
  Tier 1 (UniProt-native): 5,032 unique ECs
  Tier 2 (Pangenome):      3,783 unique ECs
  Tier 3 (Curated):        4,780 unique ECs
  RAST:                    2,439 unique ECs
  Combined:                5,428 unique ECs
  In bridge:               5,124 / 6,100 (84.0%)

EC overlap between tiers (intersection sizes):
  T1 ∩ T2: 3,770
  T1 ∩ T3: 4,698
  T2 ∩ T3: 3,687
  T1 ∩ T2 ∩ T3: 3,683

Tier-exclusive ECs:
  T1 only: 247
  T2 only: 9
  T3 only: 78


## 3. Reaction-Level Coverage Analysis

In [7]:
all_rxns = tier1_rxns | tier2_ec_rxns | tier3_rxns | rast_rxns

print(f'Reaction coverage by tier:')
print(f'  Tier 1 (UniProt-native): {len(tier1_rxns):,} / {len(balanced_ids):,} ({100*len(tier1_rxns)/len(balanced_ids):.1f}%)')
print(f'  Tier 2 (Pangenome EC):   {len(tier2_ec_rxns):,} / {len(balanced_ids):,} ({100*len(tier2_ec_rxns)/len(balanced_ids):.1f}%)')
print(f'  Tier 3 (Curated):        {len(tier3_rxns):,} / {len(balanced_ids):,} ({100*len(tier3_rxns)/len(balanced_ids):.1f}%)')
print(f'  RAST:                    {len(rast_rxns):,} / {len(balanced_ids):,} ({100*len(rast_rxns)/len(balanced_ids):.1f}%)')
print(f'  Combined (all tiers):    {len(all_rxns):,} / {len(balanced_ids):,} ({100*len(all_rxns)/len(balanced_ids):.1f}%)')
print(f'  Not reached:             {len(balanced_ids - all_rxns):,} ({100*len(balanced_ids - all_rxns)/len(balanced_ids):.1f}%)')

print(f'\nIncremental coverage (additive value when added in tier order):')
cumulative = set()
for name, rxn_set in [('Tier 1', tier1_rxns), ('Tier 2', tier2_ec_rxns), ('Tier 3', tier3_rxns), ('RAST', rast_rxns)]:
    new = rxn_set - cumulative
    cumulative |= rxn_set
    print(f'  + {name}: {len(new):,} new -> {len(cumulative):,} cumulative ({100*len(cumulative)/len(balanced_ids):.1f}%)')

Reaction coverage by tier:
  Tier 1 (UniProt-native): 17,215 / 34,343 (50.1%)
  Tier 2 (Pangenome EC):   14,557 / 34,343 (42.4%)
  Tier 3 (Curated):        16,290 / 34,343 (47.4%)
  RAST:                    10,730 / 34,343 (31.2%)
  Combined (all tiers):    17,351 / 34,343 (50.5%)
  Not reached:             16,992 (49.5%)

Incremental coverage (additive value when added in tier order):
  + Tier 1: 17,215 new -> 17,215 cumulative (50.1%)
  + Tier 2: 18 new -> 17,233 cumulative (50.2%)
  + Tier 3: 117 new -> 17,350 cumulative (50.5%)
  + RAST: 1 new -> 17,351 cumulative (50.5%)


In [8]:
print(f'Reaction overlap (pairwise):')
print(f'  T1 ∩ T2: {len(tier1_rxns & tier2_ec_rxns):,} ({100*len(tier1_rxns & tier2_ec_rxns)/len(tier1_rxns | tier2_ec_rxns):.1f}% Jaccard)')
print(f'  T1 ∩ T3: {len(tier1_rxns & tier3_rxns):,} ({100*len(tier1_rxns & tier3_rxns)/len(tier1_rxns | tier3_rxns):.1f}% Jaccard)')
print(f'  T2 ∩ T3: {len(tier2_ec_rxns & tier3_rxns):,} ({100*len(tier2_ec_rxns & tier3_rxns)/len(tier2_ec_rxns | tier3_rxns):.1f}% Jaccard)')

print(f'\nTier-exclusive reactions:')
print(f'  T1 only: {len(tier1_rxns - tier2_ec_rxns - tier3_rxns - rast_rxns):,}')
print(f'  T2 only: {len(tier2_ec_rxns - tier1_rxns - tier3_rxns - rast_rxns):,}')
print(f'  T3 only: {len(tier3_rxns - tier1_rxns - tier2_ec_rxns - rast_rxns):,}')
print(f'  RAST only: {len(rast_rxns - tier1_rxns - tier2_ec_rxns - tier3_rxns):,}')

Reaction overlap (pairwise):
  T1 ∩ T2: 14,539 (84.4% Jaccard)
  T1 ∩ T3: 16,169 (93.3% Jaccard)
  T2 ∩ T3: 14,315 (86.6% Jaccard)

Tier-exclusive reactions:
  T1 only: 796
  T2 only: 11
  T3 only: 117
  RAST only: 1


## 4. Per-Channel Breakdown

Decompose each tier into individual channels for fine-grained attribution.

In [9]:
channels = {}

for ch in ['ec', 'brenda', 'rhea']:
    ch_ecs = set(tier1[tier1['channels'].str.contains(ch, na=False)]['ec'])
    ch_rxns = set(ec_bridge[ec_bridge['ec'].isin(ch_ecs)]['rxn_bare'])
    channels[f'T1_{ch}'] = ch_rxns

for ch in ['eggnog_ec', 'bakta_ec']:
    ch_ecs = set(tier2_ec[tier2_ec['channels'].str.contains(ch, na=False)]['ec'])
    ch_rxns = set(ec_bridge[ec_bridge['ec'].isin(ch_ecs)]['rxn_bare'])
    channels[f'T2_{ch}'] = ch_rxns

for ch in ['paperblast', 'seedclass', 'besthitmetacyc_ec']:
    ch_ecs = set(tier3[tier3['channel'] == ch]['ec'])
    ch_rxns = set(ec_bridge[ec_bridge['ec'].isin(ch_ecs)]['rxn_bare'])
    channels[f'T3_{ch}'] = ch_rxns

channels['T3_besthitmetacyc_rxnid'] = tier3_metacyc_rxns
channels['RAST'] = rast_rxns

print(f'{"Channel":<35s} {"Reactions":>10s} {"Coverage":>10s}')
print('-' * 57)
for name, rxn_set in sorted(channels.items(), key=lambda x: -len(x[1])):
    print(f'{name:<35s} {len(rxn_set):>10,} {100*len(rxn_set)/len(balanced_ids):>9.1f}%')
print('-' * 57)
all_channel_rxns = set().union(*channels.values())
print(f'{"COMBINED":<35s} {len(all_channel_rxns):>10,} {100*len(all_channel_rxns)/len(balanced_ids):>9.1f}%')

Channel                              Reactions   Coverage
---------------------------------------------------------
T1_ec                                   17,092      49.8%
T3_paperblast                           16,138      47.0%
T1_brenda                               12,239      35.6%
T1_rhea                                 12,129      35.3%
T2_bakta_ec                             11,907      34.7%
T2_eggnog_ec                            11,778      34.3%
RAST                                    10,730      31.2%
T3_seedclass                             8,828      25.7%
T3_besthitmetacyc_ec                     7,112      20.7%
T3_besthitmetacyc_rxnid                    950       2.8%
---------------------------------------------------------
COMBINED                                17,351      50.5%


## 5. Evidence Integration Table

Build a per-reaction summary: for each balanced reaction, which channels provide evidence.

In [10]:
rxn_evidence = pd.DataFrame({'rxn_bare': sorted(balanced_ids)})

for name, rxn_set in channels.items():
    rxn_evidence[name] = rxn_evidence['rxn_bare'].isin(rxn_set)

channel_cols = [c for c in rxn_evidence.columns if c != 'rxn_bare']
rxn_evidence['n_channels'] = rxn_evidence[channel_cols].sum(axis=1)
rxn_evidence['has_tier1'] = rxn_evidence[['T1_ec', 'T1_brenda', 'T1_rhea']].any(axis=1)
rxn_evidence['has_tier2'] = rxn_evidence[['T2_eggnog_ec', 'T2_bakta_ec']].any(axis=1)
rxn_evidence['has_tier3'] = rxn_evidence[['T3_paperblast', 'T3_seedclass', 'T3_besthitmetacyc_ec', 'T3_besthitmetacyc_rxnid']].any(axis=1)
rxn_evidence['has_rast'] = rxn_evidence['RAST']
rxn_evidence['n_tiers'] = rxn_evidence[['has_tier1', 'has_tier2', 'has_tier3', 'has_rast']].sum(axis=1)
rxn_evidence['any_evidence'] = rxn_evidence['n_channels'] > 0

def assign_confidence(row):
    if row['n_tiers'] >= 3:
        return 'high'
    elif row['n_tiers'] == 2:
        return 'medium'
    elif row['has_tier1']:
        return 'medium'
    elif row['n_channels'] > 0:
        return 'low'
    else:
        return 'none'

rxn_evidence['confidence'] = rxn_evidence.apply(assign_confidence, axis=1)

print(f'Evidence integration table: {len(rxn_evidence):,} balanced reactions')
print(f'\nConfidence distribution:')
conf_dist = rxn_evidence['confidence'].value_counts()
for conf, ct in conf_dist.items():
    print(f'  {conf:<8s}: {ct:>6,} ({100*ct/len(rxn_evidence):.1f}%)')

print(f'\nReactions by number of supporting tiers:')
tier_dist = rxn_evidence['n_tiers'].value_counts().sort_index()
for n, ct in tier_dist.items():
    print(f'  {n} tiers: {ct:>6,} ({100*ct/len(rxn_evidence):.1f}%)')

print(f'\nReactions by number of channels:')
ch_dist = rxn_evidence['n_channels'].value_counts().sort_index()
for n, ct in ch_dist.items():
    print(f'  {n} channels: {ct:>6,} ({100*ct/len(rxn_evidence):.1f}%)')

Evidence integration table: 34,343 balanced reactions

Confidence distribution:
  none    : 16,992 (49.5%)
  high    : 14,470 (42.1%)
  medium  :  2,752 (8.0%)
  low     :    129 (0.4%)

Reactions by number of supporting tiers:
  0 tiers: 16,992 (49.5%)
  1 tiers:    925 (2.7%)
  2 tiers:  1,956 (5.7%)
  3 tiers:  3,925 (11.4%)
  4 tiers: 10,545 (30.7%)

Reactions by number of channels:
  0 channels: 16,992 (49.5%)
  1 channels:    860 (2.5%)
  2 channels:    460 (1.3%)
  3 channels:  1,111 (3.2%)
  4 channels:  1,654 (4.8%)
  5 channels:  3,298 (9.6%)
  6 channels:  1,547 (4.5%)
  7 channels:  1,616 (4.7%)
  8 channels:  1,856 (5.4%)
  9 channels:  4,248 (12.4%)
  10 channels:    701 (2.0%)


## 6. Save & Summary

In [11]:
rxn_evidence.to_parquet(f'{DATA_DIR}/evidence_integration_summary.parquet', index=False)
print(f'Saved evidence_integration_summary.parquet: {len(rxn_evidence):,} reactions')

Saved evidence_integration_summary.parquet: 34,343 reactions


In [12]:
covered = rxn_evidence['any_evidence'].sum()
not_covered = len(rxn_evidence) - covered

print('=' * 60)
print('NB06 EVIDENCE INTEGRATION SUMMARY')
print('=' * 60)
print(f'\nTotal balanced reactions: {len(balanced_ids):,}')
print(f'  With evidence:    {covered:,} ({100*covered/len(balanced_ids):.1f}%)')
print(f'  Without evidence: {not_covered:,} ({100*not_covered/len(balanced_ids):.1f}%)')
print(f'\nCoverage by tier (standalone):')
print(f'  Tier 1 (UniProt): {rxn_evidence["has_tier1"].sum():,} ({100*rxn_evidence["has_tier1"].mean():.1f}%)')
print(f'  Tier 2 (Pangeno): {rxn_evidence["has_tier2"].sum():,} ({100*rxn_evidence["has_tier2"].mean():.1f}%)')
print(f'  Tier 3 (Curated): {rxn_evidence["has_tier3"].sum():,} ({100*rxn_evidence["has_tier3"].mean():.1f}%)')
print(f'  RAST:             {rxn_evidence["has_rast"].sum():,} ({100*rxn_evidence["has_rast"].mean():.1f}%)')
print(f'\nConfidence:')
for conf in ['high', 'medium', 'low', 'none']:
    ct = (rxn_evidence['confidence'] == conf).sum()
    print(f'  {conf:<8s}: {ct:>6,} ({100*ct/len(rxn_evidence):.1f}%)')
print(f'\nSaved: evidence_integration_summary.parquet')
print(f'\nNext: NB07 -- RAST validation (precision/recall per channel)')

NB06 EVIDENCE INTEGRATION SUMMARY

Total balanced reactions: 34,343
  With evidence:    17,351 (50.5%)
  Without evidence: 16,992 (49.5%)

Coverage by tier (standalone):
  Tier 1 (UniProt): 17,215 (50.1%)
  Tier 2 (Pangeno): 14,557 (42.4%)
  Tier 3 (Curated): 16,290 (47.4%)
  RAST:             10,730 (31.2%)

Confidence:
  high    : 14,470 (42.1%)
  medium  :  2,752 (8.0%)
  low     :    129 (0.4%)
  none    : 16,992 (49.5%)

Saved: evidence_integration_summary.parquet

Next: NB07 -- RAST validation (precision/recall per channel)
